# RL Constrained CQL Training

This notebook:
1. Loads offline RL tensors from `data/processed/rl_tensors_2022_2023_constrained.npz`
2. Loads CQL hyperparameters from `configs/model.yaml` and `configs/training.yaml`
3. Trains a dueling CQL using `src/rl/cql.py`
4. Saves the trained model to `models/`

## 1. Imports & Paths

In [ ]:
import os
import sys
from pathlib import Path
import torch
import yaml

PROJECT_ROOT = Path(os.getcwd()).resolve().parent
sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
from src.rl.cql import (
    load_cql_training_config,
    train_cql,
    BullpenOfflineDataset,
    RLDatasetConfig,
)
from src.ope.offline_eval_cql import (
    OfflineEvalConfig,
    load_model_and_dataset,
    evaluate_td_error_full_mse,
    direct_policy_value_estimate,
    compute_policy_behavior_stats,
    compute_q_distributions,
    summarize_policy_behavior_stats,
    summarize_q_distributions,
)

## 2. Configurations

In [ ]:
DATA_DIR = PROJECT_ROOT / "data"
PROC_DIR = DATA_DIR / "processed"
CONFIG_DIR = PROJECT_ROOT / "configs"
MODELS_DIR = PROJECT_ROOT / "models"

MODELS_DIR.mkdir(parents=True, exist_ok=True)

YEAR_TAG = "2022_2023"
RL_TENSORS_PATH = PROC_DIR / f"rl_tensors_{YEAR_TAG}_constrained.npz"
MODEL_CFG_PATH = CONFIG_DIR / "model.yaml"
TRAIN_CFG_PATH = CONFIG_DIR / "training.yaml"
MODEL_OUT_PATH = MODELS_DIR / f"constrained_cql_model_{YEAR_TAG}.pth"

print("RL tensors:", RL_TENSORS_PATH)
print("Model config:", MODEL_CFG_PATH)
print("Training config:", TRAIN_CFG_PATH)
print("Model output:", MODEL_OUT_PATH)

## 3. Load Dataset & Build Model

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

train_cfg = load_cql_training_config(
    model_config_path=MODEL_CFG_PATH,
    data_path=RL_TENSORS_PATH,
    device=device,
)

train_cfg

ds = BullpenOfflineDataset(
    RLDatasetConfig(
        data_path=train_cfg.data_path,
        device=train_cfg.device,
    )
)

print("Dataset size:", len(ds))
print("State dim:", ds.state_dim)
print("Num actions:", ds.num_actions)
print("H (next hitters window):", ds.H)
print("R (max relievers per team):", ds.R)

## 4. Create Dueling CQL Model + Trainer
This calls train_dqn(train_cfg), which:

* loads BullpenOfflineDataset from train_cfg.data_path
* splits into train/val by train_cfg.val_fraction
* trains a dueling CQL with a target network
* logs TD-error periodically using evaluate_td_error in cql.py

In [ ]:
constrained_cql_model = train_cql(train_cfg)

## 5. Save trained model weights

In [ ]:
torch.save(constrained_cql_model.state_dict(), MODEL_OUT_PATH)
MODEL_OUT_PATH

## Offline Policy Evaluation (OPE)
Now we use src/ope/offline_eval.py to:

* load the saved model and dataset
* compute:
    * Mean Squared TD Error (MSTE)
    * Direct Q-based value of the greedy policy
    * Action agreement with the logged policy

In [ ]:
ope_cfg = OfflineEvalConfig(
    model_config_path=MODEL_CFG_PATH,
    model_path=MODEL_OUT_PATH,
    tensors_path=RL_TENSORS_PATH,
    device=device,
    batch_size=2048,
    gamma=train_cfg.gamma,
)

eval_model, eval_ds, eval_loader = load_model_and_dataset(ope_cfg)

print("Eval dataset size:", len(eval_ds))
print("State dim:", eval_ds.state_dim)
print("Num actions:", eval_ds.num_actions)

## 6. Mean Squared TD Error (MSTE)
This is the mean squared Bellman residual over the full dataset. It reuses evaluate_td_error from cql.py under the hood, passing model as both the online and target networks.

In [ ]:
mste = evaluate_td_error_full_mse(
    model=eval_model,
    loader=eval_loader,
    gamma=ope_cfg.gamma,
    device=ope_cfg.device,
)

print(f"Mean Squared TD Error (MSTE): {mste:.6f}")

## 7. Direct Q-based value estimate (FQE-style Direct Method)
For each state s: - compute Q(s, a) for all actions - mask unavailable actions - take greedy action a* = argmax_a Q(s, a) - define V_hat(s) = Q(s, a*)

Then average V_hat(s) across the dataset as an estimate of V(pi_greedy).

In [ ]:
dm_value = direct_policy_value_estimate(
    model=eval_model,
    loader=eval_loader,
    device=ope_cfg.device,
)

print(f"Direct Q-based value estimate (V(pi_greedy)): {dm_value:.6f}")

## 8. Policy Behavior Stats and Q distributions
How often does the greedy CQL action (respecting availability mask) match the logged (historical) action from the dataset?

In [ ]:
# New distributional metrics
policy_stats = compute_policy_behavior_stats(eval_model, eval_loader, device=ope_cfg.device)

q_stats = compute_q_distributions(eval_model, eval_loader, device=ope_cfg.device)

## 9. Summary

In [ ]:
print("========= FINAL CQL EVALUATION RESULTS =========")
print(f"TD Error (MSTE):              {mste:.6f}")
print(f"Direct Q-based V(pi_greedy):  {dm_value:.6f}")
summarize_policy_behavior_stats(policy_stats)
summarize_q_distributions(q_stats)

In [ ]:
import numpy as np
from pathlib import Path

npz = np.load(Path("../data/processed/rl_tensors_2022_2023.npz"))

for key in ["reward_folded"]:
    x = npz[key]
    print(key, "shape:", x.shape)
    print(
        key,
        "mean:", float(x.mean()),
        "std:", float(x.std()),
        "min:", float(x.min()),
        "max:", float(x.max()),
    )